In [1]:
import os, re, pickle, numpy as np

RAW_DIR = "./raw_npy120"
OUT_PKL = "ntu120_3d_test.pkl"

pat = re.compile(r"^(S\d{3}C\d{3}P\d{3}R\d{3}A(\d{3}))\.skeleton\.npy$")


def load_dict(path):
    arr = np.load(path, allow_pickle=True)
    return arr.item() if isinstance(arr, np.ndarray) and arr.dtype == object else arr


def build_ann(npy_path):
    base = os.path.basename(npy_path)
    m = pat.match(base)
    if m is None:
        raise ValueError(f"Unexpected filename: {base}")
    seq_id = m.group(1)        # SxxxCxxxPxxxRxxxAxxx
    label_id = int(m.group(2)) # A###

    d = load_dict(npy_path)
    xyz = d["skel_body0"]      # (T, V, 3)
    T, V, _ = xyz.shape

    keypoint = xyz[None, ...].astype("float32")                 # (1, T, V, 3)
    keypoint_score = np.ones((1, T, V, 1), dtype="float32")     # (1, T, V, 1)

    ann = {
        "start_frame": 0,
        "end_frame": int(T - 1),
        "pos_label": None,
        "frame_dir": seq_id,
        "img_shape": [1080, 1920],
        "original_shape": [1080, 1920],
        "total_frames": int(T),
        "keypoint": keypoint.tolist(),
        "keypoint_score": keypoint_score.tolist(),
        "source": "ntu",
        "label": label_id,
    }
    return ann


# pick first 10 skeleton files
all_files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".skeleton.npy"))
first10 = all_files[:10]

anns = []
for fname in first10:
    ann = build_ann(os.path.join(RAW_DIR, fname))
    anns.append(ann)

split = {
    "train": [a["frame_dir"] for a in anns[:5]],
    "val":   [a["frame_dir"] for a in anns[5:10]],
}

final_obj = {"split": split, "annotations": anns}

with open(OUT_PKL, "wb") as f:
    pickle.dump(final_obj, f, protocol=4)

print("saved:", OUT_PKL)
print("train:", len(split["train"]), "val:", len(split["val"]))
print("sample keys:", list(anns[0].keys()))
print("sample frame_dir/label/frames:", anns[0]["frame_dir"], anns[0]["label"], anns[0]["total_frames"])

saved: ntu120_3d_test.pkl
train: 5 val: 5
sample keys: ['start_frame', 'end_frame', 'pos_label', 'frame_dir', 'img_shape', 'original_shape', 'total_frames', 'keypoint', 'keypoint_score', 'source', 'label']
sample frame_dir/label/frames: S001C001P001R001A001 1 103


In [ ]:
import pickle

with open("ntu120_3d_test.pkl", "rb") as f:
    obj = pickle.load(f)

print("top-level keys:", obj.keys())
print("split sizes:", {k: len(v) for k, v in obj["split"].items()})
print("num annotations:", len(obj["annotations"]))

sample = obj["annotations"][0]
print("annotation keys:", list(sample.keys()))
print("frame_dir:", sample["frame_dir"])
print("label:", sample["label"])
print("total_frames:", sample["total_frames"])

T = len(sample["keypoint"][0])
V = len(sample["keypoint"][0][0])
C = len(sample["keypoint"][0][0][0])
print("keypoint shape (N, T, V, C):", 1, T, V, C)

top-level keys: dict_keys(['split', 'annotations'])
split sizes: {'train': 5, 'val': 5}
num annotations: 10
annotation keys: ['start_frame', 'end_frame', 'pos_label', 'frame_dir', 'img_shape', 'original_shape', 'total_frames', 'keypoint', 'keypoint_score', 'source', 'label']
frame_dir: S001C001P001R001A001
label: 1
total_frames: 103
keypoint shape (N, T, V, C): 1 103 25 3


: 